# 03 — XGBoost

The original `XGboost.ipynb` was never finished: it used a random 80/20 split (leaking future weeks into training), then separately a row-order "time-based" split that — because the raw file is grouped by division, not sorted by date — was actually training on divisions A-through-most and testing on the last few divisions alphabetically, not on held-out time at all.

This notebook finishes it properly: the same adstock/saturation features as notebooks 01-02, the same shared time-based holdout, and both the spend-only and `sales_lag1` variants for a fair three-way comparison against LightGBM.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import xgboost as xgb
from mmm import data as D
from mmm.eval import metrics

df = D.load_panel()
df["month"] = df["calendar_week"].dt.month
df["div_code"] = df["division"].astype("category")
df["sales_lag1"] = df.groupby("division")["sales"].shift(1)

train_weeks, test_weeks = D.time_split_weeks(df)
tr_mask = df["calendar_week"].isin(train_weeks)
te_mask = df["calendar_week"].isin(test_weeks)
y = df[D.TARGET]

media_feat = D.build_transformed_features(df, theta=0.5, gamma_frac=0.25, train_weeks=train_weeks)
base_feat = pd.concat([media_feat, df[D.CONTROL_VARS], df[["month", "div_code"]]], axis=1)

xgb_params = dict(n_estimators=500, learning_rate=0.03, max_depth=4, subsample=0.8,
                   colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42,
                   enable_categorical=True)


## Spend-only model

In [2]:
m_spend = xgb.XGBRegressor(**xgb_params)
m_spend.fit(base_feat.loc[tr_mask], y.loc[tr_mask])
pred_spend = m_spend.predict(base_feat.loc[te_mask])
metrics_spend = metrics(y.loc[te_mask], pred_spend)
print("XGBoost (spend-only) holdout:", metrics_spend)

importance = pd.Series(m_spend.feature_importances_, index=base_feat.columns).sort_values(ascending=False)
importance


XGBoost (spend-only) holdout: {'R2': 0.830396016127689, 'RMSE': 69464.19284050746, 'MAPE': 0.24363297582688498}


spend_google_sat       0.400921
spend_email_sat        0.211354
spend_facebook_sat     0.201818
div_code               0.069960
month                  0.058824
spend_affiliate_sat    0.023680
organic_views          0.021018
paid_views             0.012424
dtype: float32

## With `sales_lag1` (comparison only, same leakage caveat as notebook 02)

In [3]:
feat_lag = base_feat.copy()
feat_lag["sales_lag1"] = df["sales_lag1"]
valid = feat_lag["sales_lag1"].notna()

m_lag = xgb.XGBRegressor(**xgb_params)
m_lag.fit(feat_lag.loc[tr_mask & valid], y.loc[tr_mask & valid])
pred_lag = m_lag.predict(feat_lag.loc[te_mask & valid])
metrics_lag = metrics(y.loc[te_mask & valid], pred_lag)
print("XGBoost (with sales_lag1) holdout:", metrics_lag)


XGBoost (with sales_lag1) holdout: {'R2': 0.8796325084848872, 'RMSE': 58519.0913360848, 'MAPE': 0.19767281207219334}


In [4]:
comparison = pd.DataFrame({"spend_only": metrics_spend, "with_sales_lag1": metrics_lag}).T
comparison.to_csv("../reports/xgboost_metrics.csv")
importance.to_csv("../reports/xgboost_importance_spend_only.csv")
comparison


,R2,RMSE,MAPE
spend_only,0.830396,69464.192841,0.243633
with_sales_lag1,0.879633,58519.091336,0.197673


XGBoost lands within a couple points of LightGBM on both variants (see notebook 05 for the full cross-model table) — consistent, not a fluke of one library's defaults.